# Average and Median Spectrograms of Saturn's Radio flux using over entire Cassini mission duration.

In [1]:
import numpy as np
import pandas as pd

import xarray as xr
from scipy.io import readsav


from tfcat import TFCat

import matplotlib.colors as mp_colors
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

C:\Users\Local Admin\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
def get_sav_data(year):
    """Select sav data for a chosen year.
    
    Parses through an SKR year file to obtain the time, frequency and flux data.
    
    Parameters
    ----------
    year: int or str
        Year of the file wanted.
    
    Returns
    -------
    time: numpy.array
        Time series in 3 minute step.

    freq: numpy.array
        Midpoint values of Cassini frequency bins.

    flux: numpy.array (freq.shape, time.shape)
        Magnetic flux values for the chosen year.
    """

    skr_raw_fp = '../data/raw/SKR_raw'

    file_skr = skr_raw_fp + f'/SKR_{year}_CJ.sav'
    raw_skr = readsav(file_skr)
    flux, time_doy, freq = raw_skr['s'].copy(), raw_skr['t'], raw_skr['f']
    flux[flux == 0] = np.nan  # replace 0 with nans

    time = (time_doy * 24 * 3600).astype('timedelta64[s]') + np.datetime64('1997') - np.timedelta64(1, 'D') + np.timedelta64(1, 's') # SAV time correction
    time = time.astype('datetime64[m]')
    time = time.astype('datetime64[s]')

    return time, freq, flux

In [3]:
ephemeris_fp = '../data/calculated/20040101000000_20170915115700_ephemeris.csv'

Read the polygon's masked flux

In [4]:
times, fluxes = [], []
for year in range(2004, 2018):
    time, _, flux = get_sav_data(year)

    times.append(time)
    fluxes.append(flux)

_, freq, _ = get_sav_data(2004)

freq = freq.astype(float)
time = np.concatenate(times, axis=0)
flux = np.concatenate(fluxes, axis=1)

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/SKR_raw/SKR_2017_CJ.sav'

Create a panda dataframe with frequency values as columns

In [ ]:
df = pd.DataFrame(flux.T, columns=freq)
df['Universal Time'] = time
df

Read the ephemeris data as a pandas dataframe

In [ ]:
df_ephemeris = pd.read_csv(ephemeris_fp)
df_ephemeris = df_ephemeris.rename(columns={'start': 'Universal Time'})  # Rename datetime column to Universal Time
df_ephemeris["Universal Time"] = np.array(df_ephemeris['Universal Time'], dtype='datetime64')  # Convert strings to datetime64 format
df_ephemeris

Merge both dataframes on their Universal time column

In [ ]:
df_all = df.merge(df_ephemeris, on='Universal Time')

We can now bin the values with respect to local time (subLST column)

In [ ]:
bins = np.arange(0, 24.2, 0.2)  # 0 to 24hr with an interval of 12 min
df_all['local_time'] = pd.cut(df_all['subLST'], bins=bins) # Create and add bin column to the entire dataframe
df_all

Group values by bins and take their mean/median per frequency per bin

In [ ]:
df_mean = df_all.groupby('local_time').mean()
df_median = df_all.groupby('local_time').median()

The first 48 columns are all the frequency-flux values

In [ ]:
mean_flux = df_mean.iloc[:, :48].to_numpy()
median_flux = df_median.iloc[:, :48].to_numpy()

Plots

In [ ]:
fig = plt.figure(figsize=(10,5), dpi=100)
ax = fig.add_subplot()

norm = mp_colors.LogNorm(vmin=1e-22, vmax=1e-19)
cbar = ax.pcolormesh(bins[:-1], freq, mean_flux.T, norm=norm, cmap='plasma')

fig.colorbar(cbar, label=r'Flux [Log($W.m^{-2}.Hz^{-1}$)]', extend='both')

ax.set_xticks(bins[:-1:10])
ax.set_xticks(bins[:-1], minor=True)
ax.xaxis.set_major_formatter('{x:.0f}:00')

ax.set_yscale('log')
ax.set_xlabel('Local Time [hr]')
ax.set_ylabel('Frequency [kHz]');
#ax.set_title('Polygon-selected Mean Magnetic Flux per frequency channel per local time over entire Cassini duration');

In [ ]:
fig = plt.figure(figsize=(10,5), dpi=100)
ax = fig.add_subplot()

norm = mp_colors.LogNorm()
cbar = ax.pcolormesh(bins[:-1], freq, median_flux.T, norm=norm, cmap='plasma')

fig.colorbar(cbar, label=r'Flux [Log($W.m^{-2}.Hz^{-1}$)]')

ax.set_xticks(bins[:-1:10])
ax.set_xticks(bins[:-1], minor=True)
ax.xaxis.set_major_formatter('{x:.0f}:00')

ax.set_yscale('log')
ax.set_xlabel('Local Time [hr]')
ax.set_ylabel('Frequency [kHz]');
#ax.set_title('Polygon-selected Median Magnetic Flux per frequency channel per local time over entire Cassini duration');